# 00 — Setup & Data Loading

Common data loading for all hypothesis notebooks. Run this notebook first, or import from `common_loader.py`.

**Exclusions:** Subjects 154, 197, 208 (calibration outliers)
**N = 290 after exclusions**

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import ast
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import pearsonr, ttest_rel, ttest_1samp, levene, linregress
from scipy.special import expit
from pathlib import Path

%matplotlib inline
plt.rcParams.update({
    'figure.dpi': 120, 'font.size': 10,
    'axes.spines.right': False, 'axes.spines.top': False,
})

DATA_DIR = Path("/workspace/data/exploratory_350/processed/stage5_filtered_data_20260320_191950")
OUT_DIR = Path("/workspace/results/stats/full_analysis")
EXCLUDE = [154, 197, 208]

print("Libraries loaded.")

In [ ]:
# Load all data files
beh_choice = pd.read_csv(DATA_DIR / "behavior.csv")
beh_choice = beh_choice[~beh_choice['subj'].isin(EXCLUDE)].copy()
beh_choice['T_round'] = beh_choice['threat'].round(1)
beh_choice['T_H'] = beh_choice['distance_H'].map({1: 5.0, 2: 7.0, 3: 9.0})
beh_choice['effort_reqT'] = beh_choice['effort_H'] * beh_choice['T_H'] - 0.4 * 5.0
beh_choice['trial_number'] = beh_choice.groupby('subj').cumcount() + 1
beh_choice['current_score'] = beh_choice.groupby('subj')['choice'].cumsum().shift(1, fill_value=0)

beh_rich = pd.read_csv(DATA_DIR / "behavior_rich.csv", low_memory=False)
beh_rich = beh_rich[~beh_rich['subj'].isin(EXCLUDE)].copy()
beh_rich['T_round'] = beh_rich['threat'].round(1)
beh_rich['actual_dist'] = beh_rich['startDistance'].map({5: 1, 7: 2, 9: 3})
beh_rich['is_heavy'] = (beh_rich['trialCookie_weight'] == 3.0).astype(int)
beh_rich['actual_req'] = np.where(beh_rich['is_heavy'] == 1, 0.9, 0.4)
beh_rich['enc_t'] = pd.to_numeric(beh_rich['encounterTime'], errors='coerce')
beh_rich['strike_t'] = pd.to_numeric(beh_rich['strike_time'], errors='coerce')
beh_rich['is_attack'] = beh_rich['isAttackTrial'].astype(int)

feelings = pd.read_csv(DATA_DIR / "feelings.csv")
feelings = feelings[~feelings['subj'].isin(EXCLUDE)].copy()

psych = pd.read_csv(DATA_DIR / "psych.csv")
psych = psych[~psych['subj'].isin(EXCLUDE)].copy()

# Load model parameters (from full pipeline)
params = pd.read_csv(OUT_DIR / "part1_params_full.csv")

print(f"Choice trials: {len(beh_choice)} ({beh_choice['subj'].nunique()} subjects)")
print(f"All trials: {len(beh_rich)} ({beh_rich['subj'].nunique()} subjects)")
print(f"Feelings: {len(feelings)}")
print(f"Psych: {len(psych)}")
print(f"Params: {len(params)} (cols: {list(params.columns)})")